# 列表

{{ video_embed | replace("%%VID%%", "x8oLIEtSRBs")}}

OCaml 列表是一系列具有相同类型的值。它们
以单链表实现。列表在语言中是一等公民：
语言为轻松创建和使用列表提供了特殊支持。
这是 OCaml 与许多其他函数式语言共享的特性。
Python 这样的主流命令式语言如今也有这种支持。
也许这是因为程序员喜欢把列表作为语言的一等组成部分来直接使用，
而不必通过库来操作它们（像 C 和 Java 那样）。

## 构建列表

{{ video_embed | replace("%%VID%%", "I9u4kFPM7YI")}}

**语法。** 构建列表有三种语法形式：
```ocaml
[]
e1 :: e2
[e1; e2; ...; en]
```
空列表写作 `[]`，读作“nil”，这个名字来自
Lisp。给定一个列表 `lst` 和元素 `elt`，我们可以把 `elt` 加到 `lst` 前面：
写作 `elt :: lst`。双冒号运算符读作“cons”，这个名字
它来自 Lisp 中的一个运算符，该运算符 <u>cons</u>tructs 内存中的对象。
“Cons”也可以用作动词，例如“我会把一个元素 cons 到列表上”。
列表的第一个元素通常称为它的“头部”，其余元素（如果有的话）
称为它的*尾部*。

cons 运算符 `::` 是右结合的，这意味着 `e1 :: e2 :: e3` 是
理解为 `e1 :: (e2 :: e3)`，而不是 `(e1 :: e2) :: e3`。

方括号语法很方便，但没有必要。任意列表
`[e1; e2; ...; en]` 可以用更原始的 nil 和
cons 语法：`e1 :: e2 :: ... :: en :: []`。当一种好用的语法可以
用语言中更原始的语法定义出来时，我们称这种
好用的语法为*语法糖*：它让语言变得“更甜”。
把这种语法糖转换为更原始的语法称为“去糖”。

因为列表的元素可以是任意表达式，所以列表可以是
嵌套的深度任意深，例如 `[[[]]; [[1; 2; 3]]]`。

**动态语义。**

- `[]` 本身已经是一个值。
- 如果 `e1` 计算得到 `v1`，且 `e2` 计算得到 `v2`，那么 `e1 :: e2` 计算得到 `v1 :: v2`。

由这些规则以及方括号语法如何去糖化，我们得到以下推导规则：

- 如果对所有 `i` ∈ `1..n` 都有 `ei` 计算得到 `vi`，那么 `[e1; ...; en]` 计算得到 `[v1; ...; vn]`。

在求值规则中反复写"计算得到"开始变得枯燥。让我们引入一个更简洁的记号：我们用 `e ==> v` 表示"e 计算得到 v"。注意 `==>` 不是 OCaml 语法的一部分，而是我们用来描述语言的元语言符号。用这个记号，我们可以改写上面的后两个规则：

- 如果 `e1 ==> v1`，且 `e2 ==> v2`，那么 `e1 :: e2 ==> v1 :: v2`。
- 如果对所有 `i` ∈ `1..n` 都有 `ei ==> vi`，那么 `[e1; ...; en] ==> [v1; ...; vn]`。

**静态语义。**

列表的所有元素必须具有相同的类型。如果元素类型是 `t`，那么列表的类型就是 `t list`。你应该从右到左读这样的类型：`t list` 是一个 `t` 的列表，`t list list` 是一个 `t` 的列表的列表，等等。这里的 `list` 本身不是一个类型：没有办法构造一个仅为 `list` 类型的 OCaml 值。相反，`list` 是一个*类型构造器*：给定一个类型，它产生一个新的类型。例如，给定 `int`，它产生类型 `int list`。你可以把类型构造器想象成作用于类型而非值的函数。

类型检查规则：

- `[] : 'a list`
- 如果 `e1 : t` 且 `e2 : t list`，那么 `e1 :: e2 : t list`（若对优先级困惑，后者表示 `(e1 :: e2) : t list`）。

在 `[]` 的规则中，记住 `'a` 是一个类型变量：它代表一个未知的类型。所以空列表是一个元素类型未知的列表。如果我们在它前面 cons 一个 `int`，比如 `2 :: []`，编译器就会推断出这个特定列表的 `'a` 必须是 `int`。但如果在另一个地方我们在它前面 cons 一个 `bool`，比如 `true :: []`，编译器会推断出这个特定列表的 `'a` 必须是 `bool`。

## 访问列表

{{ video_embed | replace("%%VID%%", "AkrlDpHN_zE")}}

```{note}
上面链接的视频中也使用了记录和元组作为例子。这些内容会在本书的 [后面部分](records_tuples) 介绍。
```

{{ video_embed | replace("%%VID%%", "sO9wxUxajS4")}}

实际上，构造列表只有两种方法：nil 和 cons。所以如果我们想把列表分解成它的组成部分，就必须说明两种情况：列表为空时做什么，以及列表非空时（即，一个元素 cons 到另一个列表上时）做什么。我们用一种叫*模式匹配*的语言特性来实现这一点。

下面是使用模式匹配计算列表总和的示例：

In [ ]:
let rec sum lst =
  match lst with
  | [] -> 0
  | h :: t -> h + sum t

这个函数会取得输入 `lst`，并检查它是否具有与空列表相同的形状。
如果是，就返回 0。否则，如果它匹配列表形状 `h :: t`，
就让 `h` 成为 `lst` 的第一个元素，让 `t` 成为 `lst` 的其余元素，
并返回 `h + sum t`。这里的变量名表示“头部”和“尾部”，
是一种常见习惯；当然，如果愿意，我们也可以使用其他名字。
另一种常见习惯是：

In [ ]:
let rec sum xs =
  match xs with
  | [] -> 0
  | x :: xs' -> x + sum xs'

也就是说，输入列表命名为 xs（读作 EX-uhs），头元素命名为
x，尾部命名为 xs'（读作 EX-uhs prime）。

从语法上讲，没有必要使用这么多行来定义 `sum`。我们可以
全部在一行完成：

In [ ]:
let rec sum xs = match xs with | [] -> 0 | x :: xs' -> x + sum xs'

或者，请注意 `with` 之后的第一个 `|` 是可选的，无论有多少个
我们使用的行，我们也可以写：

In [ ]:
let rec sum xs = match xs with [] -> 0 | x :: xs' -> x + sum xs'

多行格式是我们在本书中通常使用的格式，因为它有助于
人眼能更好地理解语法。OCaml 代码格式化工具，
不过，只要代码很短，就会转向单行格式
足以适合仅一条线。

这是使用模式匹配计算列表长度的另一个示例：

In [ ]:
let rec length lst =
  match lst with
  | [] -> 0
  | h :: t -> 1 + length t

请注意我们实际上并不需要变量 `h` 在右侧
模式匹配。当我们想要指示模式中存在某些值时
无需实际命名，我们可以编写 `_` （下划线字符）：

In [ ]:
let rec length lst =
  match lst with
  | [] -> 0
  | _ :: t -> 1 + length t

该函数实际上是作为 OCaml 标准库 `List` 的一部分内置的
模块。它的名字是`List.length`。该“点”符号表示
名为 `List` 的模块内名为 `length` 的函数，很像点
许多其他语言中使用的符号。

这是第三个示例，它将一个列表附加到
另一个列表：

In [ ]:
let rec append lst1 lst2 =
  match lst1 with
  | [] -> lst2
  | h :: t -> h :: append t lst2

例如，`append [1; 2] [3; 4]` 是 `[1; 2; 3; 4]`。这个函数其实就是
可用作内置运算符 `@`，因此我们可以编写
`[1; 2] @ [3; 4]`。

{{ video_embed | replace("%%VID%%", "VDRTatjSl0E")}}

作为最后一个例子，我们可以编写一个函数来确定是否
列表为空：

In [ ]:
let empty lst =
  match lst with
  | [] -> true
  | h :: t -> false

但是有一种更好的方法可以在不进行模式匹配的情况下编写相同的函数：

In [ ]:
let empty lst =
  lst = []

请注意上面的所有递归函数与通过以下方法进行证明有何相似之处
自然数归纳法：每个自然数要么是 0，要么是 1
大于其他一些自然数 $n$，因此归纳证明具有
0 的基本情况和 $n + 1$ 的归纳情况。同样，我们所有的函数
有一个空列表的基本情况和一个包含列表的递归情况
比另一个列表多一个元素。这种相似性并非偶然。有一个
归纳与递归之间的深厚关系；我们将探讨这一点
本书后面将详细介绍这种关系。

顺便说一句，有两个库函数 `List.hd` 和 `List.tl` 会返回
列表的头部和尾部。但在惯用的 OCaml 中，不建议直接把这些函数
应用到列表上。问题在于，如果把它们应用到空列表，它们会抛出异常，
而你必须记得处理这个异常。相反，你应该使用模式匹配：
这样你会被迫至少处理空列表和非空列表两种情况，
从而避免异常，让程序更加健壮。

## （不）改变列表

列表是不可变的。没有办法把列表中的某个元素
改成另一个值。相反，OCaml 程序员会从旧列表创建新列表。
例如，假设我们想编写一个返回相同列表的函数
作为其输入列表，但把第一个元素（如果有）加一。
我们可以这样做：
```ocaml
let inc_first lst =
  match lst with
  | [] -> []
  | h :: t -> h + 1 :: t
```

现在你可能会担心这样是否浪费空间。毕竟，
编译器至少有两种方法可以实现上述代码：

1. 在模式匹配到 cons 时，复制整个尾部列表 `t` 来创建新列表，
使得使用的内存量增加
   数量与 `t` 的长度成比例。

2. 在旧列表和新列表之间共享尾列表 `t`，这样
使用的内存量不会增加&mdash;超出一块额外的内存
   存储 `h + 1` 所需的内存。

事实上，编译器执行的是后者。所以没必要担心。
编译器实现共享非常安全的原因正是
列表元素是不可变的。如果它们是可变的，那么我们就必须
担心我的列表是否与你的列表共享，
以及我所做的更改是否会在你的列表中可见。所以不变性使得
更容易推理代码，并使编译器可以安全地执行
执行优化。

## 与列表的模式匹配

我们在上面看到了如何使用模式匹配来访问列表。让我们更仔细地
观察这个语言特性。

**语法。**
```ocaml
match e with
| p1 -> e1
| p2 -> e2
| ...
| pn -> en
```

每个子句 `pi -> ei` 称为模式匹配的*分支*或*情况*。
整个模式匹配中的第一个竖线是可选的。

这里的 `p` 是一种新的语法形式，称为*模式*。目前，有一个模式
可能是：

* 变量名称，例如 `x`
* 下划线字符 `_`，称为 *通配符*
* 空列表 `[]`
* `p1 :: p2`
* `[p1; ...; pn]`

变量名称在模式中不得出现多次。例如，
模式 `x :: x` 是非法的。通配符可以出现任意多次。

随着我们更多地了解 OCaml 中可用的数据结构，我们将扩展
模式的可能性。

**动态语义。**

{{ video_embed | replace("%%VID%%", "sz72NP4u4DQ")}}

模式匹配涉及两个相互关联的任务：确定模式是否
匹配一个值，并确定该值的哪些部分应该关联
模式中的变量名称。前一个任务直观上是关于
确定模式和值是否具有相同的*形状*。后一个任务
是关于确定模式引入的*变量绑定*。对于
例如，考虑以下代码：

In [ ]:
match 1 :: [] with
| [] -> false
| h :: t -> h >= 1 && List.length t = 0

在求值第二个分支的右侧时，`h` 绑定到 `1`
并且 `t` 绑定到 `[]`。让我们写 `h->1` 来表示变量绑定：
`h` 具有值 `1`；这不是一段 OCaml 语法，而是一个
我们用来推理语言的符号。所以产生的变量绑定
第二个分支将是 `h->1, t->[]`。

使用该表示法，以下是模式何时匹配值的定义，并且
匹配的绑定产生：

* 模式 `x` 匹配任何值 `v` 并生成变量绑定
  `x->v`.

* 模式 `_` 匹配任何值并且不产生绑定。

* 模式 `[]` 与值 `[]` 匹配并且不生成任何绑定。

* 如果 `p1` 匹配 `v1` 并生成一组 $b_1$ 绑定，并且如果 `p2` 匹配
`v2` 并生成一组 $b_2$ 绑定，然后 `p1 :: p2` 匹配 `v1 :: v2`
  并生成绑定集 $b_1 \cup b_2$ 。注意 `v2` 必须是一个列表
  （因为它位于 `::` 的右侧）并且可以具有任意长度：0
  元素、1 个元素或多个元素。请注意，联合 $b_1 \cup b_2$
  单独绑定同一个变量时，绑定永远不会出现问题
  在 $b_1$ 和 $b_2$ 中，因为语法限制没有变量
  名称可能在模式中出现多次。

* 如果对 `1..n` 中的所有 `i`，`pi` 与 `vi` 匹配并产生集合
$b_i$ 的绑定，然后 `[p1; ...; pn]` 匹配 `[v1; ...; vn]` 并产生
  绑定集 $\bigcup_i b_i$ 。请注意，此模式指定
  列表的长度必须准确。

现在我们可以说明如何求值 `match e with p1 -> e1 | ... | pn -> en`：

* 将 `e` 计算为值 `v`。

* 尝试将 `v` 与 `p1` 匹配，然后与 `p2` 匹配，依此类推，按顺序
它们出现在匹配表达式中。

* 如果 `v` 与任何模式都不匹配，则求值
匹配表达式会引发 `Match_failure` 异常。我们还没有讨论过
  OCaml 中的异常，但你肯定在其他语言中熟悉它们
  语言。我们将在本章结尾处回到异常情况，之后
  我们已经介绍了 OCaml 中的一些其他内置数据结构。

* 否则，让 `pi` 成为第一个匹配的模式，并让 $b$ 成为匹配的模式。
通过将 `v` 与 `pi` 匹配而生成的变量绑定。

* 将这些绑定替换为 `ei` 内的 $b$，生成新表达式 `e'`。

* 将 `e'` 求值为值 `v'`。

* 整个匹配表达式的结果是`v'`。

例如，下面展示如何求值这个匹配表达式：

In [ ]:
match 1 :: [] with
| [] -> false
| h :: t -> h = 1 && t = []

* `1 :: []` 已经是一个值。

* `[]` 与 ``1 :: []`` 不匹配。

* `h :: t` 确实匹配 `1 :: []` 并生成变量绑定
{`h->1`,`t->[]`}，因为：

  - `h` 匹配 `1` 并生成变量绑定 `h->1`。

  - `t` 匹配 `[]` 并生成变量绑定 `t->[]`。

* 将 {`h->1`,`t->[]`} 替换为 `h = 1 && t = []` 内
生成一个新表达式 `1 = 1 && [] = []`。

* 求值 `1 = 1 && [] = []` 会产生值 `true`。我们省略了
此处说明该事实的理由，但它是根据其他求值规则得出的
  用于内置运算符和函数应用程序。

* 所以整个匹配表达式的结果是`true`。

**静态语义。**

* 如果 `e : ta` 且对于所有 `i`，则认为 `pi : ta` 和 `ei : tb`，
然后`(match e with p1 -> e1 | ... | pn -> en) : tb`。

该规则依赖于能够判断模式是否具有特定类型。
像往常一样，类型推断在这里发挥作用。OCaml 编译器推断
任何模式变量的类型以及所有出现的通配符
模式。至于列表模式，它们具有与以下相同的类型检查规则：
列出表达式。

**额外的静态检查。**

{{ video_embed | replace("%%VID%%", "aLQJpk9vXD4")}}

除了类型检查规则之外，编译器还有另外两项检查
对每个匹配表达式执行的操作。

首先，**详尽性：**编译器检查以确保存在
足够的模式以保证其中至少有一个与表达式匹配
`e`，无论该表达式在运行时的值是什么。这确保了
程序员没有忘记任何分支。例如，下面的函数
将导致编译器发出警告：

In [ ]:
let head lst = match lst with h :: _ -> h

通过向程序员提出警告，编译器正在帮助
程序员防范 `Match_failure` 异常的可能性
运行时。

```{note}
抱歉，上面的单元格的输出如何被分成许多行
HTML。当前是 [open issue with JupyterBook][issue]，框架
用于构建本书。

[issue]: https://github.com/executablebooks/jupyter-book/issues/973
```

其次，**未使用的分支：**编译器检查是否有任何
分支永远无法匹配，因为之前的分支之一是
保证成功。例如，下面的函数将导致编译器
发出警告：

In [ ]:
let rec sum lst =
  match lst with
  | h :: t -> h + sum t
  | [ h ] -> h
  | [] -> 0

第二个分支未使用，因为第一个分支会匹配
第二个分支能匹配的任何内容。

未使用的匹配情况通常表明程序员编写了其他内容
比他们的意图。因此，通过提出该警告，编译器正在提供帮助
程序员检测代码中潜在的错误。

以下是导致未使用匹配情况警告的最常见错误之一。
理解它也是检查你是否真正理解
匹配表达式的动态语义：

In [ ]:
let length_is lst n =
  match List.length lst with
  | n -> true
  | _ -> false

程序员原本想表达：如果 `lst` 的长度等于 `n`，那么
该函数返回 `true`，否则返回 `false`。但事实上
该函数*总是*返回 `true`。为什么？因为模式变量 `n` 是
与函数参数 `n` 不同。假设 `lst` 的长度为 5。
那么模式匹配就变成：`match 5 with n -> true | _ -> false`。`n`
是否匹配 5？是的，根据上面的规则：变量模式匹配任何值。
这里产生绑定 `n->5`。然后求值会把该绑定应用到
`true`，也就是用 5 替换 `true` 中所有出现的 `n`。
没有这样的出现。因此求值完成，结果就是
`true`。

程序员真正想写的是：

In [ ]:
let length_is lst n =
  match List.length lst with
  | m -> m = n

或者更好：

In [ ]:
let length_is lst n =
  List.length lst = n

## 深度模式匹配

模式可以嵌套。  这样做可以让你的代码深入了解
列表的结构。  例如：

* `_ :: []` 与所有列表仅匹配一个元素

* `_ :: _` 匹配所有至少包含一个元素的列表

* `_ :: _ :: []` 匹配所有包含两个元素的列表

* `_ :: _ :: _ :: _` 匹配所有至少包含三个元素的列表

## 立即匹配

{{ video_embed | replace("%%VID%%", "VgVP8Tin6yY")}}

当你有一个立即与其最终模式匹配的函数时
争论，有一个很好的语法糖可以用来避免写作
额外的代码。这是一个例子：而不是

In [ ]:
let rec sum lst =
  match lst with
  | [] -> 0
  | h :: t -> h + sum t

你可以写

In [ ]:
let rec sum = function
  | [] -> 0
  | h :: t -> h + sum t

`function` 一词是一个关键字。请注意，我们可以省略该行
包含 `match` 以及从未使用过的参数名称
除了那条线之外的任何其他地方。但在这种情况下，这一点尤其重要
函数的规范注释，用于记录该参数的内容
应该是，因为代码不再给它一个描述性的名称。

## OCamldoc 和列表语法

OCamldoc 是一个类似于 Javadoc 的文档生成器。它提取注释
从源代码生成 HTML（以及其他输出格式）。
生成 `List` 模块的 [标准库网页文档][std-web]
由该模块的 [标准库源代码][std-src] 中的 OCamldoc 提供，
例如。

```{warning}
OCamldoc 中有一个使用方括号的语法约定，
很容易和列表混淆。

在 OCamldoc 注释中，源代码用方括号括起来。那段代码
会以等宽字体呈现，并在输出 HTML 中进行语法高亮。
本例中的方括号并不表示列表。
```

例如，这是标准库源代码中 `List.hd` 的注释
代码：
```ocaml
(** Return the first element of the given list. Raise
   [Failure "hd"] if the list is empty. *)
```
`[Failure "hd"]` 并不意味着包含异常的列表
`Failure "hd"`。相反，它意味着将表达式 `Failure "hd"` 排版为
源代码，如你所见[here][std-web]。

当你想将列表作为以下内容的一部分进行讨论时，这可能会变得特别令人困惑
文档。例如，我们可以这样重写该注释：
```ocaml
(** [hd lst] returns the first element of [lst].
    Raises [Failure "hd"] if [lst = []]. *)
```
在 `[lst = []]` 中，外部方括号表示源代码作为
注释，而内部方括号表示空列表。

[std-web]: https://ocaml.org/api/List.html
[std-src]: https://github.com/ocaml/ocaml/blob/trunk/stdlib/list.mli

## 列表推导式

一些语言，包括 Python 和 Haskell，有一种称为
*推导式*的语法，允许用类似数学中集合推导式的方式编写列表。
推导式最早的例子似乎来自
函数式语言 NPL，设计于 1977 年。

OCaml 没有内置的推导式语法支持。虽然有些
开发了扩展，但似乎不再支持任何扩展。初级
通过推导式完成的主要任务（过滤掉一些元素，以及
转换其他元素）实际上已经得到*高阶
编程*，我们将在后面的章节中学习，以及管道运算符，
我们已经了解到了。因此，推导式提供的额外语法
从来没有真正需要过。

## 尾递归

回想一下，如果一个函数递归地调用自身，那么它就是“尾递归”，但是
递归调用返回后不执行任何计算，并且
立即将其递归调用的值返回给调用者。考虑
这两个实现，`sum` 和 `sum_tr` 对列表求和：

In [ ]:
let rec sum (l : int list) : int =
  match l with
  | [] -> 0
  | x :: xs -> x + (sum xs)

let rec sum_plus_acc (acc : int) (l : int list) : int =
  match l with
  | [] -> acc
  | x :: xs -> sum_plus_acc (acc + x) xs

let sum_tr : int list -> int =
  sum_plus_acc 0

请观察上面的 `sum` 和 `sum_tr` 函数之间的以下区别：
在`sum`函数中，不是尾递归，递归调用之后
返回其值，我们向其添加 `x` 。在尾部递归 `sum_tr` 中，或者更确切地说
在`sum_plus_acc`中，递归调用返回后，我们立即返回
值无需进一步计算。

如果你要在很长的列表上编写函数，尾递归就变成了
对性能很重要。因此，当你可以选择使用
尾递归与非尾递归函数，你可能最好使用
在很长的列表上使用尾递归函数以实现空间效率。
因此，List 模块记录了哪些函数是尾递归的
哪些不是。

但这并不意味着尾递归实现严格来说更好。
例如，尾递归函数可能更难阅读。 （考虑
`sum_plus_acc`。）此外，在某些情况下实现尾递归
函数需要进行预处理或后处理以反转
列表。在中小型列表上，反转列表的开销（两者
及时并为反向列表分配内存）可以使
尾递归版本时间效率较低。什么是“小”与“大”
在这里？这很难说，但根据
[`List` 模块的标准库文档][list]。

[list]: https://ocaml.org/api/List.html

这是一个有用的尾递归函数，可以生成长列表：

In [ ]:
(** [from i j l] is the list containing the integers from [i] to [j],
    inclusive, followed by the list [l].
    Example:  [from 1 3 [0] = [1; 2; 3; 0]] *)
let rec from i j l = if i > j then l else from i (j - 1) (j :: l)

(** [i -- j] is the list containing the integers from [i] to [j], inclusive. *)
let ( -- ) i j = from i j []

let long_list = 0 -- 1_000_000

值得研究 `--` 的定义，以说服自己
你了解（i）它是如何工作的以及（ii）为什么它是尾递归的。

你将来可能会决定再次创建这样的列表。而不是
必须记住这个定义在哪里，并且必须将其复制到你的
代码，这是使用内置库创建相同列表的简单方法
函数：

In [ ]:
List.init 1_000_000 Fun.id

表达式 `List.init len f` 创建列表 `[f 0; f 1; ...; f (len - 1)]`，
如果 `len` 大于 10,000，它会递归地尾部执行此操作。函数
`Fun.id` 只是识别函数 `fun x -> x`。